# Business Analytics & Statistical Analysis

##Exploratory Data Analysis (EDA) - Retail sales
###Name: Carlos Munoz
###Perform Exploratory Data Analysis on dataset the sample Google Play store.

####Dataset: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

#Directory, Library, Data

In [ ]:
%cd /content/drive/MyDrive/Projects/SQL/Google Traffic

In [ ]:
#Install needed libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load the data
google_df = pd.read_csv('google_play_store.csv')
google_df.head()

#Data preparation & Cleaning

##Describe the data

In [ ]:
google_df.info()

#Check for Outliers

##Handling Outliers

##Remove Outliers

# Data Preparation & Cleaning

## Describe the data

In [ ]:
google_df.info()

## Handle missing values and fix column types

In [ ]:
# Rating has 1,474 nulls — drop them for analysis
google_df['Rating'] = pd.to_numeric(google_df['Rating'], errors='coerce')
google_df = google_df[google_df['Rating'] <= 5.0].copy()

# Clean Installs column (remove '+' and ',' then cast to int)
google_df['Installs_clean'] = (
    google_df['Installs']
    .str.replace('[+,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)

# Clean Reviews column
google_df['Reviews'] = pd.to_numeric(google_df['Reviews'], errors='coerce')

# Encode Type as binary
google_df['IsFree'] = (google_df['Type'] == 'Free').astype(int)

print(f"Clean dataset: {google_df.shape[0]:,} apps across {google_df['Category'].nunique()} categories")

# Load into SQLite3

In [ ]:
# Create in-memory SQLite database and ingest the cleaned DataFrame
conn = sqlite3.connect(':memory:')
google_df.to_sql('apps', conn, if_exists='replace', index=False)
print("Table loaded into SQLite.")

# SQL Analysis

## Top 10 categories by average rating

In [ ]:
query = """
SELECT Category,
       ROUND(AVG(Rating), 2)    AS avg_rating,
       COUNT(*)                  AS app_count
FROM   apps
WHERE  Rating IS NOT NULL
GROUP  BY Category
ORDER  BY avg_rating DESC
LIMIT  10
"""
top_rated = pd.read_sql_query(query, conn)
print(top_rated.to_string(index=False))

In [ ]:
top_rated.plot(kind='barh', x='Category', y='avg_rating', figsize=(10, 6),
               color='steelblue', legend=False)
plt.title('Top 10 App Categories by Average Rating', fontsize=14)
plt.xlabel('Average Rating')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

## Top 10 most-installed categories

In [ ]:
query2 = """
SELECT Category,
       SUM(Installs_clean) / 1000000.0  AS total_installs_M
FROM   apps
WHERE  Installs_clean IS NOT NULL
GROUP  BY Category
ORDER  BY total_installs_M DESC
LIMIT  10
"""
top_installs = pd.read_sql_query(query2, conn)
print(top_installs.to_string(index=False))

In [ ]:
top_installs.plot(kind='bar', x='Category', y='total_installs_M', figsize=(12, 5),
                  color='darkorange', legend=False)
plt.title('Top 10 App Categories by Total Installs (Millions)', fontsize=14)
plt.xlabel('Category')
plt.ylabel('Total Installs (M)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Free vs. Paid app breakdown by category

In [ ]:
query3 = """
SELECT Type,
       COUNT(*)               AS app_count,
       ROUND(AVG(Rating), 2)  AS avg_rating
FROM   apps
WHERE  Type IN ('Free', 'Paid')
GROUP  BY Type
"""
free_paid = pd.read_sql_query(query3, conn)
print(free_paid.to_string(index=False))

In [ ]:
# Top categories for paid apps
query4 = """
SELECT Category,
       COUNT(*) AS paid_count
FROM   apps
WHERE  Type = 'Paid'
GROUP  BY Category
ORDER  BY paid_count DESC
LIMIT  10
"""
paid_cats = pd.read_sql_query(query4, conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Free vs Paid count
axes[0].bar(free_paid['Type'], free_paid['app_count'], color=['#2196F3', '#FF5722'])
axes[0].set_title('Free vs. Paid App Count')
axes[0].set_ylabel('Number of Apps')

# Top categories for paid apps
axes[1].barh(paid_cats['Category'], paid_cats['paid_count'], color='#9C27B0')
axes[1].set_title('Top 10 Categories for Paid Apps')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Content rating distribution

In [ ]:
query5 = """
SELECT "Content Rating"   AS content_rating,
       COUNT(*)             AS app_count,
       ROUND(AVG(Rating), 2) AS avg_rating
FROM   apps
GROUP  BY "Content Rating"
ORDER  BY app_count DESC
"""
content = pd.read_sql_query(query5, conn)
print(content.to_string(index=False))

In [ ]:
content.plot(kind='bar', x='content_rating', y='app_count', figsize=(10, 5),
             color='teal', legend=False)
plt.title('App Count by Content Rating', fontsize=14)
plt.xlabel('Content Rating')
plt.ylabel('Number of Apps')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Rating distribution across all apps

In [ ]:
sns.histplot(google_df['Rating'], bins=20, kde=True, color='steelblue')
plt.title('Distribution of App Ratings', fontsize=14)
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Summary

## Key Findings

- **10,841 apps** across **34 categories** after cleaning and removing invalid ratings
- **Free apps dominate:** over 92% of apps are free; paid apps concentrate in Family, Medical, and Tools
- **Top installed categories:** Communication, Social, and Video Players lead by total install volume
- **Highest-rated categories:** Events, Education, and Art & Design consistently earn the best average ratings
- **Data quality:** The Rating column required preprocessing — 1,474 null values and a handful of erroneous entries above 5.0 were removed before aggregation
